# Claims Data Profiling & Initial Analysis

This notebook is used to inspect the three vendor claim datasets before building
the harmonization pipeline.

In [25]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

ROOT = Path("..")
DATA = ROOT / "data"

SOURCE_A = DATA / "source_a_claims.csv.xlsx"
SOURCE_B = DATA / "source_b_claims.csv.xlsx"
SOURCE_C = DATA / "source_c_claims.csv.xlsx"
DICTIONARY = DATA / "dx_dictionary.csv.xlsx"

In [26]:
source_a = pd.read_excel(SOURCE_A)
source_b = pd.read_excel(SOURCE_B)
source_c = pd.read_excel(SOURCE_C)
dictionary = pd.read_excel(DICTIONARY)


In [27]:
def profile_table(df, name):
    return pd.DataFrame({
        "Dataset": [name],
        "Rows": [len(df)],
        "Columns": [len(df.columns)],
        "Duplicate Rows": [df.duplicated().sum()]
    })


overview = pd.concat([
    profile_table(source_a, "Source A"),
    profile_table(source_b, "Source B"),
    profile_table(source_c, "Source C"),
    profile_table(dictionary, "Dictionary")
], ignore_index=True)

overview

,Dataset,Rows,Columns,Duplicate Rows
0,Source A,26004,26,0
1,Source B,53891,19,0
2,Source C,24186,20,0
3,Dictionary,40,3,0


In [28]:
column_comparison = pd.DataFrame({
    "Source A": pd.Series(source_a.columns),
    "Source B": pd.Series(source_b.columns),
    "Source C": pd.Series(source_c.columns)
})

column_comparison

,Source A,Source B,Source C
0,patient_id,member_id,pt_ref
1,claim_id,encounter_id,claim_ref
2,service_nbr,line_nbr,version
3,service_from_date,svc_date,seq
4,diagnosis_code_1,dx_code,date_of_service
...,...,...,...
21,primary_plan_id,NaN,NaN
22,secondary_plan_id,NaN,NaN
23,unit_of_svc_amt,NaN,NaN
24,bill_amt,NaN,NaN


### Observation

The three vendors do not use the same column names for the same information.

In [29]:
def missing_summary(df, name):
    result = df.isna().sum()
    result = result[result > 0]

    return (
        result
        .rename("Missing")
        .to_frame()
        .assign(
            Dataset=name,
            Missing_Percent=lambda x: (x["Missing"] / len(df) * 100).round(2)
        )
        .reset_index()
        .rename(columns={"index": "Column"})
    )


missing = pd.concat([
    missing_summary(source_a, "Source A"),
    missing_summary(source_b, "Source B"),
    missing_summary(source_c, "Source C"),
    missing_summary(dictionary, "Dictionary")
], ignore_index=True)

missing.sort_values(["Dataset", "Missing"], ascending=[True, False])

,Column,Missing,Dataset,Missing_Percent
7,diagnosis_code_8,25494,Source A,98.04
6,diagnosis_code_7,25422,Source A,97.76
5,diagnosis_code_6,24027,Source A,92.40
4,diagnosis_code_5,21831,Source A,83.95
3,diagnosis_code_4,18656,Source A,71.74
12,secondary_plan_id,18300,Source A,70.37
2,diagnosis_code_3,13856,Source A,53.28
11,provider_referring_id,7863,Source A,30.24
1,diagnosis_code_2,7289,Source A,28.03
0,patient_id,402,Source A,1.55


### Observation

Source A contains records with missing patient information.

Decision:

**Drop records with missing patient IDs.**

In [30]:
print("Source A")
print(source_a["patient_gender"].value_counts(dropna=False))

print("\nSource B")
print(source_b["gender"].value_counts(dropna=False))

print("\nSource C")
print(source_c["sex"].value_counts(dropna=False))

Source A
patient_gender
M      12900
F      12702
NaN      402
Name: count, dtype: int64

Source B
gender
2    26971
1    26920
Name: count, dtype: int64

Source C
sex
Male      12125
Female    12061
Name: count, dtype: int64


### Observation


| Source | Representation |
|---|---|
| Source A | M / F |
| Source B | 1 / 2 |
| Source C | Male / Female |

Decision:

**Normalize all sources to M/F.**

In [31]:
print("Source A diagnosis columns:")
print([
    col for col in source_a.columns
    if col.startswith("diagnosis_code_")
])

print("\nSource B diagnosis column:")
print(source_b["dx_code"].dropna().head(20).tolist())

print("\nSource C diagnosis examples:")
print(source_c["diagnosis_codes"].dropna().head(20).tolist())

Source A diagnosis columns:
['diagnosis_code_1', 'diagnosis_code_2', 'diagnosis_code_3', 'diagnosis_code_4', 'diagnosis_code_5', 'diagnosis_code_6', 'diagnosis_code_7', 'diagnosis_code_8']

Source B diagnosis column:
['K5900', 'C50911', 'M797', 'T889', 'G4733', 'C3490', 'T889', 'N184', 'Q998', 'Z5111', 'E039', 'R0602', 'E669', 'I4891', 'R9720', 'C7951', 'R9720', 'F329', 'R9720', 'I4891']

Source C diagnosis examples:
['F411|C61|E669|C50.911', 'N18.3| C3490 |L03.115|M17.11', 'M79.7', 'T889|i2510', ' Q99.8 |R51.9|E78.5', 'I4891|E78.5', 'Z85.46', 'b37.0', 'K219|M79.7', 'H35.31', 'Z99.89|I50.32', 'R519', 'J45909| I50.32 |R6889', 'm17.11|H2513', 'R519', 'R97.20', 'n390|J44.9', 'E785', 'R51.9|C34.90', 'Z8546']


### Observation

Diagnosis data has three different structures.

Source A:

```text
diagnosis_code_1
diagnosis_code_2
...
diagnosis_code_8

Source B :

dx_code

Source C:

one column - F411|C61|E669|

Decision:

Convert all diagnosis representations into individual diagnosis rows.

In [32]:
source_a_dx = (
    source_a[
        [col for col in source_a.columns if col.startswith("diagnosis_code_")]
    ]
    .stack()
    .dropna()
    .astype(str)
)

source_b_dx = source_b["dx_code"].dropna().astype(str)

source_c_dx = source_c["diagnosis_codes"].dropna().astype(str)

print("Source A sample:")
print(source_a_dx.head(10).tolist())

print("\nSource B sample:")
print(source_b_dx.head(10).tolist())

print("\nSource C sample:")
print(source_c_dx.head(10).tolist())

Source A sample:
['E03.9', 'Q99.8', 'G47.33', 'T88.9', 'Z51.11', 'Z99.89', 'N18.4', 'B37.0', 'M54.5', 'D64.9']

Source B sample:
['K5900', 'C50911', 'M797', 'T889', 'G4733', 'C3490', 'T889', 'N184', 'Q998', 'Z5111']

Source C sample:
['F411|C61|E669|C50.911', 'N18.3| C3490 |L03.115|M17.11', 'M79.7', 'T889|i2510', ' Q99.8 |R51.9|E78.5', 'I4891|E78.5', 'Z85.46', 'b37.0', 'K219|M79.7', 'H35.31']


In [33]:
multi_dx = source_c_dx[source_c_dx.str.contains(r"\|", regex=True)]

print("Count:", len(multi_dx))
print(multi_dx.head(10).tolist())

Count: 14951
['F411|C61|E669|C50.911', 'N18.3| C3490 |L03.115|M17.11', 'T889|i2510', ' Q99.8 |R51.9|E78.5', 'I4891|E78.5', 'K219|M79.7', 'Z99.89|I50.32', 'J45909| I50.32 |R6889', 'm17.11|H2513', 'n390|J44.9']


### Observation

Source C contains multiple diagnosis codes inside one field using `|`.

Example:

```text
I10|D64.9|K21.9

In [34]:

samples = pd.DataFrame({
    "Original": [
        "E11.9",
        "e11.9",
        " R51.9 ",
        "m17.11",
        "C34.90"
    ]
})

samples["Normalized"] = (
    samples["Original"]
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

samples

,Original,Normalized
0,E11.9,E119
1,e11.9,E119
2,R51.9,R519
3,m17.11,M1711
4,C34.90,C3490


### Observation

The same diagnosis can appear with:

- lowercase letters
- spaces
- dots

Decision:

```text
Trim → Uppercase → Remove dots

In [35]:
source_c["claim_ref"].nunique(), source_c["version"].value_counts().sort_index()


(20001,
 version
 1    20001
 2     3525
 3      660
 Name: count, dtype: int64)

In [36]:
version_counts = (
    source_c
    .groupby("claim_ref")["version"]
    .nunique()
)

print("Total claims:", source_c["claim_ref"].nunique())
print("Claims with multiple versions:", (version_counts > 1).sum())

Total claims: 20001
Claims with multiple versions: 3525


In [37]:
multi_version_claims = (
    version_counts[version_counts > 1]
    .sort_values(ascending=False)
)

multi_version_claims.head(10)

claim_ref
C0002305    3
C0015526    3
C0015530    3
C0009036    3
C0009029    3
C0017758    3
C0017750    3
C0012641    3
C0002847    3
C0005453    3
Name: version, dtype: int64

### Observation

Source C contains multiple versions of the same claim.

There are:

- 24,186 original rows
- 3,525 claims with multiple versions

This means Source C contains revisions of existing claims.

Decision:

**Keep the latest version of each claim.**

The superseded records will be tracked separately in the pipeline.

In [44]:
source_c[
    source_c["claim_ref"] == "C9000000"
].sort_values("version")

,pt_ref,claim_ref,version,seq,date_of_service,diagnosis_codes,yob,sex,zip_3,service_place,facility,claim_category,npi_rendering,npi_referring,npi_billing,plan_1,plan_2,amount_unit,amount_billed,source_system
14778,P00042,C9000000,1,24185,2022-12-27,i10|D649| K21.9,1949,Male,331,81,2,P,1617916890,NaN,1083867151,PLN5276,PLN1376,248.67,4307.52,SRC_C
8839,P00042,C9000000,2,24186,2022-12-27,F32.9,1949,Male,331,11,3,P,1870588458,1.124320e+09,1797368225,PLN4412,PLN3828,325.56,874.74,SRC_C


In [39]:
date_summary = pd.DataFrame({
    "Dataset": ["Source A", "Source B", "Source C"],
    "Min Date": [
        source_a["service_from_date"].min(),
        source_b["svc_date"].min(),
        source_c["date_of_service"].min()
    ],
    "Max Date": [
        source_a["service_from_date"].max(),
        source_b["svc_date"].max(),
        source_c["date_of_service"].max()
    ]
})

date_summary

,Dataset,Min Date,Max Date
0,Source A,20160108,20251230
1,Source B,2016-01-04 00:00:00,2025-12-30 00:00:00
2,Source C,2016-01-03 00:00:00,2025-12-30 00:00:00


In [40]:
START_DATE = pd.Timestamp("2018-01-01")
END_DATE = pd.Timestamp("2025-02-28")

source_b_outside = (
    (source_b["svc_date"] < START_DATE) |
    (source_b["svc_date"] > END_DATE)
).sum()

source_c_outside = (
    (source_c["date_of_service"] < START_DATE) |
    (source_c["date_of_service"] > END_DATE)
).sum()

print("Source B outside range:", source_b_outside)
print("Source C outside range:", source_c_outside)

Source B outside range: 1072
Source C outside range: 487


In [41]:
source_a_dates = pd.to_datetime(
    source_a["service_from_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

source_a_outside = (
    (source_a_dates < START_DATE) |
    (source_a_dates > END_DATE)
).sum()

print("Source A outside range:", source_a_outside)

Source A outside range: 507


### Decision

Keep only service dates between:

```text
2018-01-01
2025-02-28

In [42]:

print(dictionary.head())
print()

print("Dictionary codes:", dictionary["dx_code"].nunique())
print("Dictionary rows:", len(dictionary))

  dx_code                                     dx_description icd_version
0    E119     TYPE 2 DIABETES MELLITUS WITHOUT COMPLICATIONS       ICD10
1   E1165        TYPE 2 DIABETES MELLITUS WITH HYPERGLYCEMIA       ICD10
2     I10                     ESSENTIAL PRIMARY HYPERTENSION       ICD10
3   I2510  ATHEROSCLEROTIC HEART DISEASE OF NATIVE CORONA...       ICD10
4    J449  CHRONIC OBSTRUCTIVE PULMONARY DISEASE UNSPECIFIED       ICD10

Dictionary codes: 40
Dictionary rows: 40


In [43]:
all_dx = pd.concat([
    source_a_dx,
    source_b_dx,
    source_c_dx.str.split("|").explode()
], ignore_index=True)

all_dx = (
    all_dx
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

dictionary_codes = (
    dictionary["dx_code"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

missing_from_dictionary = sorted(
    set(all_dx.dropna()) - set(dictionary_codes)
)

print("Unique diagnosis codes in claims:", all_dx.nunique())
print("Dictionary codes:", dictionary_codes.nunique())
print("Codes missing from dictionary:", len(missing_from_dictionary))

print(missing_from_dictionary)

Unique diagnosis codes in claims: 44
Dictionary codes: 40
Codes missing from dictionary: 4
['Q998', 'R6889', 'T889', 'Z9989']


### Observation

Not every diagnosis code in the claims data exists in the dictionary.

### Options

1. Drop diagnosis records without a dictionary match.
2. Keep the diagnosis and leave the description empty.

### Decision

Keep the diagnosis.

The dictionary is an enrichment source, not a validation rule. A missing
description should not remove an otherwise valid claim.

# Profiling Findings

The profiling produced the following decisions:

| Finding | Decision |
|---|---|
| Source columns differ | Standardize before combining |
| Source A has 8 diagnosis columns | Convert to rows |
| Source B has one diagnosis field | Normalize directly |
| Source C has `|` separated diagnoses | Split into rows |
| Diagnosis formats differ | Uppercase + remove dots |
| Source C has multiple claim versions | Keep latest version |
| Missing patient IDs exist | Drop |
| Dates exist outside required range | Drop |
| Gender formats differ | Normalize to M/F |
| Some codes are missing from dictionary | Keep code, description can be missing |
| Final grain required | SRC + CLAIM_ID + DIAGNOSIS_CODE |

These findings were used to design the harmonization pipeline.